# Milestone 1 — run it all from VS Code

Steps 1-2 (preprocess + train the 5-channel GAN). The GAN is launched as a **background
process**, so the kernel never freezes during the ~20-min MIOpen warmup (that freeze is
what "crashed" the kernel before). You watch progress in the monitor cell.

After the GAN finishes -> open **run_akbar_check_multi.ipynb** for step 3 (the Dice number).

## Step 1 — Preprocess (4 modalities + mask)
Runs in this kernel, ~15-20 min (loads 5 scans x ~370 subjects). Streams progress live so
the cell never looks frozen. Builds data_multi\train + data_multi\test.

In [ ]:
import subprocess, sys, os, time
PY = r"c:\Users\Tonkid\AppData\Local\Python\pythoncore-3.12-64\python.exe"
proc = subprocess.Popen([PY, "-u", r"C:\Users\Tonkid\Downloads\tstr\preprocess_paired_multi.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:          # stream live -> cell stays visibly alive
    print(line, end="", flush=True)
proc.wait()
print("\n[preprocess exit code", proc.returncode, "]")

## Step 2 — Launch the 5-channel GAN in the background

**Edit the parameters below**, then run the cell. It returns immediately; training runs in
its own process (survives the cell finishing) and writes to run_paired_multi.log.
**Do not close VS Code** while it trains (~overnight).

- `KIMG` = how long the GAN trains -- the main dial. Higher = better fakes but longer.
  1000 is a good overnight target; bump to 1500-2000 if the Dice lands short.
- The rest are the anti-collapse recipe from your 2-ch run; leave them unless you know why.

In [ ]:
# ---- GAN parameters (edit me) ----
KIMG     = 1000      # main dial: how long the GAN trains
BATCH    = 16        # images per step
LR       = 0.0015    # learning rate
R1_GAMMA = 10.0      # anti-collapse regularization
P_INIT   = 0.5       # augmentation on from step 0 (anti-collapse)
# ----------------------------------

LOG = r"C:\Users\Tonkid\Downloads\run_paired_multi.log"
# "-u" = unbuffered, so progress lines appear in the log immediately (not stuck in a buffer).
cmd = [PY, "-u", r"C:\Users\Tonkid\Downloads\run_paired_multi.py",
       "--kimg", str(KIMG), "--batch", str(BATCH), "--lr", str(LR),
       "--r1_gamma", str(R1_GAMMA), "--p_init", str(P_INIT)]
proc = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("GAN training launched, PID", proc.pid, "| kimg", KIMG)
print("log ->", LOG)
print("\nNOTE: the FIRST kimg takes ~1-2 hours (one-time MIOpen kernel search) and the log")
print("will show harmless 'MIOpenIm2d2Col' compile errors during it -- that is NORMAL, the")
print("GPU falls back to a working kernel. After the first tick it speeds up to ~50 s/kimg.")
print("Use the monitor cell below to watch real progress (it filters out the MIOpen noise).")

## Step 2b — Monitor (re-run this cell anytime)
Filters out the harmless MIOpen compile noise and shows only real status: how long it's
been running, whether it's still in warmup, real progress lines, and the latest sample.

In [ ]:
import glob, os, time, numpy as np, matplotlib.pyplot as plt
NOISE = ("MIOpen", "get_global", "errors generated", "HIPRTC", "hipoc", "index_t",
         "tid +=", "^~~", "|", "cuDNN", "benchmark_limit")
try:
    with open(LOG) as f: raw = f.read().splitlines()
except FileNotFoundError:
    raw = []
real = [ln for ln in raw if ln.strip() and not any(k in ln for k in NOISE)]
n_err = sum("MIOpenIm2d2Col" in ln for ln in raw)
age_min = (time.time() - os.path.getmtime(LOG)) / 60 if os.path.exists(LOG) else 0
started_min = (time.time() - os.path.getctime(LOG)) / 60 if os.path.exists(LOG) else 0

# is a real training tick present yet?
ticks = [ln for ln in real if any(k in ln.lower() for k in ("kimg", "tick", "sec/kimg", "loss_d"))]
print(f"running for ~{started_min:.0f} min | log last changed {age_min:.1f} min ago | "
      f"{n_err} harmless im2col msgs")
if ticks:
    print("\n-- real progress (latest) --")
    print("\n".join(ticks[-8:]))
elif started_min < 130:
    print("\nStill in first-tick WARMUP (normal, up to ~1-2 h). No real tick yet -- this is")
    print("expected, not a hang. The im2col messages are harmless. Check back later.")
else:
    print("\nWARNING: >2 h with no training tick -- that is longer than warmup should take.")
print("\n-- last non-noise lines --")
print("\n".join(real[-6:]) if real else "(nothing but warmup noise yet)")

# --- latest sample image ---
samps = sorted(glob.glob(r"C:\Users\Tonkid\Downloads\runs\paired256_multi\samples\*.png"))
if samps:
    import matplotlib.image as mpimg
    plt.figure(figsize=(10, 8)); plt.imshow(mpimg.imread(samps[-1])); plt.axis("off")
    plt.title("latest: " + os.path.basename(samps[-1])); plt.show()
else:
    print("\n(no sample image yet -- first one appears after the first tick completes)")

## When Step 2 says DONE
The log will end with `5-CHANNEL PAIRED GAN DONE`. Then:
1. Open **run_akbar_check_multi.ipynb** and run it top-to-bottom -> your 4-modality Dice vs 0.737.
2. (Optional topline) run tstr/real_baseline_multi.py the same background way.